# M2 B — Eclat: Groceries và Splice DNA

Notebook Python gọi Julia 1.9+ của nhóm, chạy từ nguồn/hash đã pin đến kết quả thật.
Mặc định bật preparation, miner, hai backend, hai ứng dụng và Java SPMF verification.
RUN_BENCHMARKS/FINAL_DELIVERY_VALIDATION dành cho M3 và mặc định tắt.
Chọn Runtime → Run all. Colab tải Julia có kiểm checksum và cài Java 21;
local cần ECLAT_JAVA trỏ vào Java >=21 nếu Java mặc định quá cũ.


In [ ]:
from pathlib import Path
import csv
import json
import tempfile
import time
import os
import sys
import hashlib
import re
import shutil
import subprocess
import urllib.request

REPO_URL = "https://github.com/phuctran33/-DM-Eclat.git"
REPO_REF = os.environ.get("ECLAT_REPO_REF", "feat/member-b-m2-applications")
RUN_PREPARATION = True
JULIA_VERSION = "1.11.4"
RUN_MINER = True
RUN_SPMF = True
RUN_OPTIMIZED = True
RUN_APPLICATIONS = True
RUN_BENCHMARKS = False
FINAL_DELIVERY_VALIDATION = False
SEED = 42

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content") / ("eclat-m2-" + str(time.time_ns()))
    if not (ROOT / "Project.toml").exists():
        if not REPO_URL.startswith("https://github.com/"):
            raise ValueError("Điền URL GitHub public thật của nhóm trước khi chạy Colab.")
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    if not shutil.which("julia"):
        archive = Path("/content/julia-runtime.tar.gz")
        runtime = Path("/content/julia-runtime")
        runtime.mkdir(exist_ok=True)
        urllib.request.urlretrieve(
            f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz",
            archive,
        )
        checksum_url = f"https://julialang-s3.julialang.org/bin/checksums/julia-{JULIA_VERSION}.sha256"
        checksum_text = urllib.request.urlopen(checksum_url, timeout=30).read().decode("utf-8")
        filename = f"julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
        expected = next(line.split()[0] for line in checksum_text.splitlines() if line.split()[-1].lstrip("*") == filename)
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Julia archive checksum mismatch"
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(runtime), "--strip-components=1"], check=True)
        os.environ["PATH"] = str(runtime / "bin") + os.pathsep + os.environ["PATH"]
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Project.toml").exists()), None)
    if ROOT is None:
        raise RuntimeError("Mở notebook từ repo root hoặc notebooks/.")

os.chdir(ROOT)
os.environ.setdefault("JULIA_DEPOT_PATH", str(ROOT / ".tools/julia-depot"))
JULIA = os.environ.get("ECLAT_JULIA") or shutil.which("julia")
if not JULIA:
    candidates = sorted((ROOT / ".tools").glob("julia-*/bin/julia.exe"))
    JULIA = str(candidates[-1]) if candidates else None
if not JULIA:
    raise RuntimeError("Cài Julia 1.9+ và thêm vào PATH khi chạy local.")

def run_julia(*arguments, env=None):
    return subprocess.run(
        [JULIA, "--startup-file=no", f"--project={ROOT}", *map(str, arguments)],
        cwd=ROOT, env={**os.environ, **(env or {})}, check=True,
    )

version = subprocess.check_output([JULIA, "--version"], text=True)
match = re.search(r"(\d+)\.(\d+)", version)
assert match and tuple(map(int, match.groups())) >= (1, 9), "Julia 1.9+ required"
assert (ROOT / "scripts/prepare_applications.jl").is_file(), "Checkout phải có code B; chọn đúng nhánh đã publish hoặc mở source B local."
run_julia("--version")
run_julia("-e", "using Pkg; Pkg.instantiate()")
# Respect an explicit executable; otherwise find an existing Java >=21 locally.
def java_major(executable):
    try:
        result = subprocess.run([str(executable), "-version"], capture_output=True,
                                text=True, timeout=15)
    except (OSError, subprocess.TimeoutExpired):
        return 0
    match = re.search(r'(?:openjdk|java)(?:\s+version)?\s+"?(\d+)(?:\.(\d+))?',
                      result.stdout + result.stderr)
    if result.returncode or not match:
        return 0
    return int(match[2]) if match[1] == "1" and match[2] else int(match[1])

JAVA = os.environ.get("ECLAT_JAVA") or shutil.which("java")
if RUN_SPMF and not IN_COLAB and not os.environ.get("ECLAT_JAVA"):
    java_candidates = [JAVA] if JAVA else []
    for pattern in ["jdk*/bin/java.exe", "java*/bin/java.exe", "jdk*/bin/java", "java*/bin/java"]:
        java_candidates.extend((ROOT / ".tools").glob(pattern))
    if os.name == "nt":
        for directory in [os.environ.get("ProgramFiles"), os.environ.get("ProgramFiles(x86)")]:
            if directory:
                for pattern in ["Java/*/bin/java.exe", "Eclipse Adoptium/*/bin/java.exe", "JetBrains/*/jbr/bin/java.exe"]:
                    java_candidates.extend(Path(directory).glob(pattern))
        java_candidates.extend(Path(ROOT.anchor).glob("CLion*/jbr/bin/java.exe"))
    JAVA = next((str(path) for path in java_candidates if java_major(path) >= 21), None)
if RUN_SPMF:
    if IN_COLAB:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "openjdk-21-jre-headless"], check=True)
        JAVA = "/usr/lib/jvm/java-21-openjdk-amd64/bin/java"
    if not JAVA or java_major(JAVA) < 21:
        raise RuntimeError("Java >=21 required for live SPMF. Set ECLAT_JAVA to its executable.")
    java_version = subprocess.run([JAVA, "-version"], capture_output=True, text=True, check=True)
    print("JAVA:", JAVA)
    print(java_version.stderr.strip())
    subprocess.run([sys.executable, "scripts/prepare_spmf.py", "--java", JAVA], check=True)
print("ROOT:", ROOT, "SEED:", SEED)
import tempfile
work_parent = ROOT / ".tools/notebook-runs"
work_parent.mkdir(parents=True, exist_ok=True)
NOTEBOOK_OUTPUT = Path(tempfile.mkdtemp(prefix="integrated-", dir=work_parent))


## B: dữ liệu và audit

Nguồn/hash và toàn bộ preprocessing đã được kiểm chéo độc lập. Cell tiếp theo
chuẩn bị dữ liệu thật; dữ liệu này chưa phải kết quả ứng dụng mined cuối cùng.


In [ ]:
if RUN_PREPARATION:
    import tempfile
    subprocess.run([sys.executable, "scripts/prepare_application_data.py", "--dataset", "all"], check=True)
    work_parent = ROOT / ".tools/notebook-runs"
    work_parent.mkdir(parents=True, exist_ok=True)
    B_OUTPUT = Path(tempfile.mkdtemp(prefix="member-b-", dir=work_parent))
    run_julia("scripts/prepare_applications.jl", "groceries", "data/applications/groceries.csv", B_OUTPUT / "groceries")
    run_julia("scripts/prepare_applications.jl", "splice", "data/applications/splice.data", B_OUTPUT / "splice")
    subprocess.run([sys.executable, "-B", "tests/verify_prepared_data.py", "--julia", JULIA,
                    "--report", str(B_OUTPUT / "preparation-report.json")], check=True)
    print("B preprocessing artifacts:", B_OUTPUT)
else:
    print("B preprocessing không chạy trong phiên này.")


## Kiểm thử chung sau tích hợp

Full suite bật miner list/bitset, rules, CLI, SPMF và toàn bộ checks B.


In [ ]:
run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1", "ECLAT_TEST_BACKENDS": "list,bitset"})
subprocess.run([sys.executable, "-B", "tests/test_application_sources.py"], check=True)
run_julia("scripts/mine.jl", "--help")


## Miner Basic và references SPMF

CLI chạy code A đã merge. References A/B chạy độc lập và có cùng hashes.
Bật RUN_SPMF nếu cần tái tạo bằng Java >=21; ECLAT_JAVA chọn executable.


In [ ]:
if RUN_MINER:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", NOTEBOOK_OUTPUT / "basic.txt",
              "--minsup", "40%", "--backend", "list")
    run_julia("scripts/compare_spmf.jl", NOTEBOOK_OUTPUT / "basic.txt", "tests/reference/spmf/basic.txt")
if RUN_SPMF:
    print("JAR SPMF đã kiểm SHA-256; live verification hai ứng dụng thực hiện ở cell tiếp theo.")


## M2: hai ứng dụng từ dữ liệu thật

Groceries: 9.835 giỏ, 169 items; minsup=0.005 (count=50), minconf=0.60.
Hai backend nhóm phải có cùng toàn bộ map, rồi sinh mọi luật đủ ngưỡng và lấy top-10 theo lift.
Splice: window 25:36, 48 IDs; giữ 3.185/3.190 mẫu. Nhãn EI/IE/N không đưa vào miner.
Minsup=0.20 riêng từng lớp; mine đầy đủ rồi hợp candidates 2–4 item và đếm lại mọi lớp.
SPMF kiểm cả Groceries và ba lớp bằng cùng absolute count, không giới hạn output.


In [ ]:
if RUN_OPTIMIZED:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", NOTEBOOK_OUTPUT / "bitset.txt",
              "--minsup", "40%", "--backend", "bitset")
    run_julia("scripts/compare_spmf.jl", NOTEBOOK_OUTPUT / "bitset.txt", "tests/reference/spmf/basic.txt")

if RUN_APPLICATIONS:
    APPLICATION_OUTPUT = NOTEBOOK_OUTPUT / "applications"
    run_julia("scripts/run_applications.jl", "--output", APPLICATION_OUTPUT)
    command = [sys.executable, "-B", "scripts/verify_application_results.py", "--output", str(APPLICATION_OUTPUT)]
    if RUN_SPMF:
        command += ["--java", JAVA]
    subprocess.run(command, check=True)
    validation = json.loads((APPLICATION_OUTPUT / "validation.json").read_text(encoding="utf-8"))
    print("Ứng dụng đã kiểm:", validation["n_rules"], "luật;", validation["n_dna_candidates"], "DNA patterns")
    def show_csv(path, limit=10):
        with path.open(encoding="utf-8", newline="") as stream:
            rows = list(csv.DictReader(stream))[:limit]
        try:
            import pandas as pd
            from IPython.display import display
            display(pd.DataFrame(rows))
        except ImportError:
            for row in rows:
                print(row)
    show_csv(APPLICATION_OUTPUT / "groceries_top10.csv")
    show_csv(APPLICATION_OUTPUT / "dna_patterns.csv")
    print("Artifacts:", APPLICATION_OUTPUT)
else:
    print("RUN_APPLICATIONS đang tắt trong phiên này.")


## Diễn giải kết quả ở cấu hình đã chốt

- Groceries có 1.001 tập phổ biến và 22 luật. Luật citrus fruit + root vegetables + whole milk → other vegetables có 57/90 giỏ thỏa hậu kiện, confidence 63,33%, lift 3,273; có thể thử bố trí/gợi ý mua kèm nhóm rau.
- Root vegetables + tropical fruit + yogurt → whole milk: 56/80 = 70%, lift 2,740. Butter + yogurt → whole milk: 92/144 = 63,89%, lift 2,500. Đây là gợi ý thử nghiệm bán chéo; không suy luận khuyến mãi gây tăng mua hàng từ dữ liệu quan sát.
- DNA có 463 candidates. (+1,G),(+2,T): EI 758/765=99,08%, IE 108/766=14,10%, N 98/1654=5,93%. (-2,A),(-1,G): EI 388/765=50,72%, IE 762/766=99,48%, N 127/1654=7,68%.
- Đây là co-occurrence mô tả trong mẫu, không phải accuracy classifier, p-value hay kết luận nhân quả sinh học. Lớp N đông hơn; dùng support riêng mỗi lớp. Cửa sổ/ngưỡng/ambiguity và việc chọn pattern trên cùng dữ liệu giới hạn khả năng khái quát.

CSV/config/run metadata là nguồn số liệu; xem docs/M2_MEMBER_B.md cho đầy đủ bảng, counts và lệnh tái lập.


## Benchmark A

Controller đã triển khai, hỗ trợ Linux/macOS; RUN_BENCHMARKS mặc định tắt.
Nếu bật, dùng output mới để giữ nguyên evidence đã publish. Hai phạm vi timing
được lưu riêng; allocation không thay peak RSS. Xem docs/experiments.md.


In [ ]:
if RUN_BENCHMARKS:
    # Real A pipeline; Java >=21, Python >=3.11. Full grid can take several minutes.
    subprocess.run([sys.executable, "scripts/prepare_spmf.py", "--java", JAVA], check=True)
    subprocess.run([sys.executable, "scripts/acquire_benchmarks.py"], check=True)
    run_julia("scripts/benchmark_runner.jl", "--java", JAVA, "--output", str(NOTEBOOK_OUTPUT / "benchmark"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-plot.txt"], check=True)
    subprocess.run([sys.executable, "scripts/plot_results.py", "--results", str(NOTEBOOK_OUTPUT / "benchmark")], check=True)
else:
    print("Benchmark pipeline đã có; RUN_BENCHMARKS đang tắt.")


## Final cold-run — B chủ trì, A review

Trước nộp: bỏ mọi TODO cần thực thi, bật đủ cờ, restart runtime và Run all.
Đặt FINAL_DELIVERY_VALIDATION=True để kiểm artifacts cơ bản; vẫn cần reviewer kiểm
chất lượng nội dung, đủ đồ thị, đúng metrics và public links. Report phải 10–15
trang IEEE tiếng Việt, paper gốc + ≥3 nguồn khác. README phải có public Colab URL.

In [ ]:
if FINAL_DELIVERY_VALIDATION:
    assert all([RUN_MINER, RUN_SPMF, RUN_OPTIMIZED, RUN_APPLICATIONS, RUN_BENCHMARKS]), "Chưa bật đủ pipeline."
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1",
                                      "ECLAT_TEST_BACKENDS": "list,bitset"})
    for relative in ["docs/Report.pdf", "results/applications/groceries_rules.csv",
                     "results/applications/dna_patterns.csv"]:
        assert (ROOT / relative).is_file(), f"Thiếu artifact: {relative}"
    print("Đã qua kiểm tra artifacts cơ bản; A/B tiếp tục rà checklist DELIVERY.md.")
else:
    print("Ứng dụng M2 đã chạy; nghiệm thu cuối còn Report.pdf và remote Colab cold-run.")